# Sentence Embedding SVC

Train a `LinearSVC` classifier on SentenceTransformer embeddings in Google Colab and log every train/validation experiment to the shared `metrics/classic_ml_experiments.csv` file used by `02_Classic_ML_Classification.ipynb`.

This notebook does not generate test predictions, does not create a submission file, and does not save a model by default.

## Dependencies

This block installs the only extra library that is not usually available in a fresh Colab runtime. Run it once at the start of the session if the imports fail or if Colab has restarted.

In [ ]:
# Run this in a fresh Colab runtime if sentence-transformers is not installed.
# %pip install -U sentence-transformers scikit-learn pandas matplotlib joblib

## Imports

This block loads the libraries used by the notebook: file paths, metrics, plotting, cached NumPy arrays, the SentenceTransformer encoder, and the `LinearSVC` classifier.

In [ ]:
from pathlib import Path
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

## Project Path

Use the repository root as the project path so the notebook runs from GitHub/local clones without Google Drive setup.


In [ ]:
project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent
project_path = project_path.resolve()

print("Project path:", project_path)


## Project Files And Output Paths

This block defines where the notebook reads the cleaned training file and where it writes shared metrics, validation diagnostics, the combined summary plot, and cached train/validation embeddings. It does not define test-data paths because this notebook does not generate final predictions or submissions.

In [ ]:
processed_dir = project_path / "datasets" / "processed"

metrics_dir = project_path / "metrics"
embeddings_dir = project_path / "embeddings" / "sentence_transformer"

train_cleaned_path = processed_dir / "train_cleaned.csv"

classic_metrics_path = metrics_dir / "classic_ml_experiments.csv"
summary_plot_path = metrics_dir / "classic_ml_summary.png"
validation_diagnostics_path = metrics_dir / "sentence_embedding_svc_validation_diagnostics.csv"

train_embeddings_path = embeddings_dir / "train_embeddings_all_minilm_l6_v2.npy"
val_embeddings_path = embeddings_dir / "validation_embeddings_all_minilm_l6_v2.npy"

metrics_dir.mkdir(parents=True, exist_ok=True)
embeddings_dir.mkdir(parents=True, exist_ok=True)

print("Processed dir:", processed_dir)
print("Shared metrics CSV:", classic_metrics_path)

## Load Cleaned Training Data

This block reads `train_cleaned.csv`, checks that the required columns exist, removes unusable rows, and validates that labels are binary. It uses `headline`, not `processed_text`, because SentenceTransformer models are trained on natural sentence-like text.

In [ ]:
if not train_cleaned_path.exists():
    raise FileNotFoundError(
        f"Missing cleaned training file: {train_cleaned_path}. "
        "Run 01_data_initialization.ipynb first."
    )

data = pd.read_csv(train_cleaned_path)

required_columns = {"label", "headline"}
missing_columns = required_columns - set(data.columns)

if missing_columns:
    raise ValueError(f"Missing columns in cleaned training data: {missing_columns}")

data = data.dropna(subset=["label", "headline"]).copy()
data["headline"] = data["headline"].astype(str).str.strip()
data = data[data["headline"] != ""]
data["label"] = data["label"].astype(int)

bad_labels = set(data["label"].unique()) - {0, 1}
if bad_labels:
    raise ValueError(f"Unexpected labels found: {bad_labels}")

print("Cleaned data shape:", data.shape)
print(data["label"].value_counts())

## Create Shared Train/Validation Split

This block creates the train/validation split used for the embedding experiment. It uses the same `random_state=42`, `test_size=0.2`, and stratification logic as the classic ML notebook so validation metrics are comparable.

In [ ]:
X = data["headline"]
y = data["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Train label ratio:")
print(y_train.value_counts(normalize=True))
print("Validation label ratio:")
print(y_val.value_counts(normalize=True))

## Load SentenceTransformer Model

This block downloads or loads `all-MiniLM-L6-v2`, a compact pretrained model that converts each headline into a dense numeric vector. The model name is saved in the experiment logs because the classifier only makes sense with embeddings from the same encoder.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
EMBEDDING_MODEL_NAME

## Encode And Cache Embeddings

This block converts text into dense embeddings. Encoding can be slow, so train and validation embeddings are cached as `.npy` files and reused on later runs when the row counts match. Test embeddings are intentionally not created in this notebook.

In [ ]:
def load_or_create_embeddings(texts, path, model, batch_size=64):
    texts = texts.astype(str)

    if path.exists():
        cached_embeddings = np.load(path)
        if cached_embeddings.shape[0] == len(texts):
            print(f"Loading cached embeddings from: {path}")
            return cached_embeddings

        print(
            "Cached embeddings row count does not match current data. "
            f"Recreating: {path}"
        )

    print(f"Creating embeddings and saving to: {path}")
    embeddings = model.encode(
        texts.tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )

    np.save(path, embeddings)
    return embeddings

In [ ]:
train_embeddings = load_or_create_embeddings(
    X_train,
    train_embeddings_path,
    embedding_model,
)

val_embeddings = load_or_create_embeddings(
    X_val,
    val_embeddings_path,
    embedding_model,
)

print("Train embeddings shape:", train_embeddings.shape)
print("Validation embeddings shape:", val_embeddings.shape)

## Train Baseline LinearSVC

This block trains the first sentence-embedding classifier. The input is already numeric because the previous block converted each headline into a vector, so `LinearSVC` can fit directly on the embedding arrays.

In [ ]:
svc_embedding_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=10000,
)

svc_embedding_model.fit(train_embeddings, y_train)

y_train_pred = svc_embedding_model.predict(train_embeddings)
y_val_pred = svc_embedding_model.predict(val_embeddings)

## Log Metrics To Shared Experiment CSV

This block builds one experiment row with the same core metric columns used by `02_Classic_ML_Classification.ipynb`, then appends it to `metrics/classic_ml_experiments.csv`. Keeping one shared CSV makes it easier to compare Bag of Words, TF-IDF, Random Forest, and SentenceTransformer experiments in one table.

In [ ]:
def append_metrics(result, csv_path=classic_metrics_path):
    result = result.copy()
    result["logged_at"] = datetime.now().isoformat(timespec="seconds")

    new_row = pd.DataFrame([result])

    if csv_path.exists():
        old_rows = pd.read_csv(csv_path)
        if "experiment_number" in old_rows.columns:
            old_rows = old_rows.drop(columns=["experiment_number"])
        updated_rows = pd.concat([old_rows, new_row], ignore_index=True, sort=False)
    else:
        updated_rows = new_row

    updated_rows.insert(0, "experiment_number", range(1, len(updated_rows) + 1))
    updated_rows.to_csv(csv_path, index=False)

    return updated_rows


def build_result(model_name, classifier, train_pred, val_pred):
    return {
        "experiment_id": datetime.now().strftime("%Y%m%d_%H%M%S_%f"),
        "model_family": "SentenceEmbeddingSVC",
        "model_name": model_name,
        "vectorizer": "SentenceTransformer",
        "classifier": "LinearSVC",
        "embedding_model": EMBEDDING_MODEL_NAME,
        "text_column": "headline",
        "train_size": len(X_train),
        "validation_size": len(X_val),
        "embedding_dimensions": train_embeddings.shape[1],
        "C": classifier.C,
        "max_iter": classifier.max_iter,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "validation_accuracy": accuracy_score(y_val, val_pred),
        "validation_precision": precision_score(
            y_val,
            val_pred,
            average="weighted",
            zero_division=0,
        ),
        "validation_recall": recall_score(
            y_val,
            val_pred,
            average="weighted",
            zero_division=0,
        ),
        "validation_f1": f1_score(
            y_val,
            val_pred,
            average="weighted",
            zero_division=0,
        ),
    }


baseline_result = build_result(
    model_name="LinearSVC_all_MiniLM_L6_v2_C_1.0_baseline",
    classifier=svc_embedding_model,
    train_pred=y_train_pred,
    val_pred=y_val_pred,
)

all_results = append_metrics(baseline_result)
all_results.tail()

## Optional Tuning

This block tries a few `C` values for `LinearSVC`. Set `RUN_TUNING = False` if you only want the baseline. When tuning runs, every `C` value is logged to the same shared experiment CSV, and the best validation-F1 classifier stays in memory for diagnostics.

In [ ]:
RUN_TUNING = True
C_VALUES = [0.25, 0.5, 2.0]

selected_model = svc_embedding_model
selected_result = baseline_result
tuning_results = []

if RUN_TUNING:
    for C in C_VALUES:
        clf = LinearSVC(
            C=C,
            random_state=42,
            max_iter=10000,
        )

        clf.fit(train_embeddings, y_train)

        train_pred = clf.predict(train_embeddings)
        val_pred = clf.predict(val_embeddings)

        result = build_result(
            model_name=f"LinearSVC_all_MiniLM_L6_v2_C_{C}",
            classifier=clf,
            train_pred=train_pred,
            val_pred=val_pred,
        )

        tuning_results.append(result)
        all_results = append_metrics(result)

        if result["validation_f1"] > selected_result["validation_f1"]:
            selected_model = clf
            selected_result = result

tuning_df = pd.DataFrame(tuning_results).sort_values(
    "validation_f1",
    ascending=False,
) if tuning_results else pd.DataFrame()

svc_embedding_model = selected_model
experiment_result = selected_result
y_train_pred = svc_embedding_model.predict(train_embeddings)
y_val_pred = svc_embedding_model.predict(val_embeddings)

print("Selected sentence embedding model:")
print(experiment_result)
tuning_df.head()

## Save One Validation Diagnostics File

This block writes one validation diagnostics CSV for the selected in-memory sentence embedding model. It keeps the original validation rows, actual labels, predicted labels, correctness flags, and error type labels so false positives and false negatives can be inspected without creating multiple separate files.

In [ ]:
validation_diagnostics_df = data.loc[X_val.index].copy()
validation_diagnostics_df["model_family"] = experiment_result["model_family"]
validation_diagnostics_df["model_name"] = experiment_result["model_name"]
validation_diagnostics_df["embedding_model"] = experiment_result["embedding_model"]
validation_diagnostics_df["prediction"] = y_val_pred
validation_diagnostics_df["actual_label_name"] = validation_diagnostics_df["label"].map({
    0: "FAKE",
    1: "REAL",
})
validation_diagnostics_df["predicted_label_name"] = validation_diagnostics_df["prediction"].map({
    0: "FAKE",
    1: "REAL",
})
validation_diagnostics_df["correct"] = (
    validation_diagnostics_df["label"] == validation_diagnostics_df["prediction"]
)

validation_diagnostics_df["error_type"] = "correct"
validation_diagnostics_df.loc[
    (validation_diagnostics_df["label"] == 0)
    & (validation_diagnostics_df["prediction"] == 1),
    "error_type",
] = "false_positive"
validation_diagnostics_df.loc[
    (validation_diagnostics_df["label"] == 1)
    & (validation_diagnostics_df["prediction"] == 0),
    "error_type",
] = "false_negative"

validation_diagnostics_df.to_csv(validation_diagnostics_path, index=False)
print(f"Saved validation diagnostics to: {validation_diagnostics_path}")
validation_diagnostics_df.head()

## Update Shared Summary Plot

This block rebuilds the shared summary plot from `metrics/classic_ml_experiments.csv`. Because it reads the combined CSV, the chart includes classic ML and sentence-embedding experiments together.

In [ ]:
all_results = pd.read_csv(classic_metrics_path)

metric_columns = [
    "train_accuracy",
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
]

for column in metric_columns:
    all_results[column] = pd.to_numeric(all_results[column], errors="coerce")

top_models = (
    all_results
    .dropna(subset=["validation_f1"])
    .sort_values("validation_f1", ascending=False)
    .head(12)
    .copy()
)
top_models["plot_label"] = (
    top_models["model_family"].astype(str)
    + " | "
    + top_models["model_name"].astype(str)
)

metric_evolution_df = all_results.copy().reset_index(drop=True)



## Optional Model Save

The notebook does not save models by default. This block is intentionally commented out. Uncomment it only if you need a reusable sentence-embedding SVC artifact. The classifier must be saved with the embedding model name because it only works with embeddings from that same model.

In [ ]:
# model_artifact = {
#     "embedding_model_name": EMBEDDING_MODEL_NAME,
#     "classifier": svc_embedding_model,
#     "label_mapping": {
#         0: "FAKE",
#         1: "REAL",
#     },
# }
# model_path = project_path / "models" / "sentence_embedding_svc" / "sentence_embedding_svc.joblib"
# model_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump(model_artifact, model_path)

## End Of Notebook

This final block lists what the notebook should create after a complete run. There should be no test prediction file, no submission file, and no saved model unless you explicitly uncommented the optional save block.

Expected outputs:

- `embeddings/sentence_transformer/train_embeddings_all_minilm_l6_v2.npy`
- `embeddings/sentence_transformer/validation_embeddings_all_minilm_l6_v2.npy`
- `metrics/classic_ml_experiments.csv`
- `metrics/classic_ml_summary.png`
- `metrics/sentence_embedding_svc_validation_diagnostics.csv`

Final test predictions and submission files should be handled only in the DistilBERT transfer learning notebook.